In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("prophet") is None:
    print("Installing prophet ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "prophet", "--quiet"], check=True
    )

import cmdstanpy

try:
    cmdstanpy.cmdstan_path()
    print("cmdstan backend already installed.")
except ValueError:
    print("Installing cmdstan backend (first run only, can take a few minutes)...")
    cmdstanpy.install_cmdstan()

print("Environment ready.")


In [ ]:
import pandas as pd
from prophet import Prophet
import numpy as np

df = pd.read_csv(
    "MOP 6 - Exercise 7 - KPI Forecasting - Sample KPI report - 30 months.csv"
)

print("Sample KPI file loaded successfully")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()


In [ ]:
# Convert Date column (explicit day-first format)

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date"])
print("After fixing dates:", len(df))


In [ ]:
# Remove duplicate Cell-Date records
df = df.sort_values(["Cell", "Date"])
df = df.drop_duplicates(subset=["Cell", "Date"], keep="first")
print("After duplicate Cell-Date removal:", len(df))


In [ ]:
# Remove rows with missing values
df = df.dropna(subset=["Date", "Cell", "Traffic_GB"])
print("After removing missing values:", len(df))


In [ ]:
# Validate KPI value ranges (based on your local rules - telecom sanity)
df = df[df["Traffic_GB"] > 0]
print("After removing non-positive traffic values:", len(df))


In [ ]:
# Sort Data (critical for time series)
df = df.sort_values(["Cell", "Date"])


In [ ]:
# Final data quality summary
print("\nFINAL DATA QUALITY SUMMARY")
print("----------------------------")
print("Total rows after validation:", len(df))
print("Number of unique cells:", df["Cell"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())
df.describe()


In [ ]:
# Save validated KPI dataset
validated_file = "Validated_KPI_Report_30_Months.csv"
df.to_csv(validated_file, index=False)
print(f"\nValidated KPI file saved as: {validated_file}")


In [ ]:
# Reload validated KPI file for downstream steps

df = pd.read_csv(validated_file)
df["Date"] = pd.to_datetime(df["Date"])
print("Validated KPI file reloaded for AI training.")
print("Rows available for modeling:", len(df))


In [ ]:
future_months = 6
forecast_rows = []
cell_models = {}  # keep fitted models + forecasts so we don't retrain for the plot below

for cell, cell_df in df.groupby("Cell"):
    prophet_df = cell_df.rename(columns={"Date": "ds", "Traffic_GB": "y"})[["ds", "y"]]

    if len(prophet_df) < 2:
        print(f"Skipping {cell}: not enough data points to fit a model.")
        continue

    model = Prophet(
        yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False
    )

    model.fit(prophet_df)

    future = model.make_future_dataframe(periods=future_months, freq="ME")

    forecast = model.predict(future)
    future_forecast = forecast.tail(future_months)

    cell_models[cell] = {"model": model, "forecast": forecast}

    for _, row in future_forecast.iterrows():
        forecast_rows.append(
            {
                "Date": row["ds"],
                "Cell": cell,
                "Forecast_Traffic_GB": round(row["yhat"], 1),
            }
        )

print(f"\nForecasted {len(cell_models)} cells, {future_months} months ahead each.")


In [ ]:
forecast_df = pd.DataFrame(forecast_rows)

forecast_df.head(10)


In [ ]:
sample_cell = df["Cell"].iloc[0]

model = cell_models[sample_cell]["model"]
forecast = cell_models[sample_cell]["forecast"]

fig = model.plot(forecast)
fig.gca().set_title(f"Traffic forecast — {sample_cell}")
fig.gca().set_xlabel("Date")
fig.gca().set_ylabel("Traffic (GB)")


In [ ]:
forecast_df.to_csv("Traffic_Forecast_Next_6_Months_Per_Cell_Prophet.csv", index=False)
print("Forecast KPI report saved successfully")
